# Timestamp Generator

## Data Loading

In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/predictive_maintenance.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


## Change Columns Name

In [2]:
column_rename_map = {
    # Identifiers
    "UDI": "unit_id",
    "Product ID": "product_id",
    # Categorical
    "Type": "engine_type",
    # Targets
    "Target": "target",
    "Failure Type": "failure_type",
    # Numeric
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque_nm",
    "Tool wear [min]": "tool_wear",
}

df = df.rename(columns=column_rename_map)
df.head()

,unit_id,product_id,engine_type,air_temp,process_temp,rpm,torque_nm,tool_wear,target,failure_type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


## Define Features Variables

In [3]:
df_preprocessed = df.copy()

original_product_id = df["product_id"].copy()

categorical_features = ["engine_type"]

numeric_features = [
    "air_temp",
    "process_temp",
    "rpm",
    "torque_nm",
    "tool_wear",
]

target = ["failure_type"]

## Generate Synthetic Timestamp

In [4]:
SEED = 42
rng = np.random.default_rng(SEED)

print(f"[INFO] Input Rows: {len(df_preprocessed)} | Unique product_id: {df_preprocessed['product_id'].nunique()}")

# Configuration
START_DATE = pd.Timestamp("2024-01-01")   # base calendar start
MAX_START_SHIFT_DAYS = 120                # each machine starts within first 4 months

# Different logging frequency per engine_type (minutes between logs)
FREQ_BY_TYPE = {
    "L": 30,   # low-quality: less frequent logging
    "M": 15,   # medium: normal
    "H": 5,    # high: very frequent logging
}

# Small random jitter around base interval
JITTER_FRACTION = 0.30 # ±30% of base interval

# Occasional bigger gaps (e.g. downtime, nights, weekends)
GAP_PROB = 0.03 # 3% of steps become a “big gap”
GAP_HOURS_MIN = 1
GAP_HOURS_MAX = 6

df_preprocessed["timestamp"] = pd.NaT

# Generate timestamps per product_id
for pid, idx in df_preprocessed.groupby("product_id").groups.items():
    sub = df_preprocessed.loc[idx].copy()

    # Use engine_type of this product (already decoded to 'L','M','H')
    eng_letter = str(sub["engine_type"].iloc[0])
    base_interval = FREQ_BY_TYPE.get(eng_letter, 15)  # default 15 if unknown type

    # Choose a realistic start datetime for this machine
    day_offset = rng.integers(0, MAX_START_SHIFT_DAYS + 1)
    start_hour = rng.integers(6, 19)  # start sometime between 06:00 and 18:00
    base_start = (
        START_DATE
        + pd.Timedelta(day_offset, "D")
        + pd.Timedelta(start_hour, "H")
    )

    # Sort rows so that higher wear (or rpm) is later in time
    if "tool_wear" in sub.columns and sub["tool_wear"].nunique() > 1:
        sub = sub.sort_values("tool_wear")
    elif "rpm" in sub.columns and sub["rpm"].nunique() > 1:
        sub = sub.sort_values("rpm")
    else:
        sub = sub.sort_index()

    n = len(sub)

    # Build cumulative minute offsets with jitter and occasional big gaps
    minutes_offsets = [0]
    for i in range(1, n):
        # Base step
        step = base_interval

        # Jitter
        jitter_amp = int(base_interval * JITTER_FRACTION)
        if jitter_amp > 0:
            jitter = rng.integers(-jitter_amp, jitter_amp + 1)
            step = max(1, step + jitter)

        # Occasional large gap (downtime)
        if rng.random() < GAP_PROB:
            step += rng.integers(GAP_HOURS_MIN * 60, GAP_HOURS_MAX * 60 + 1)

        minutes_offsets.append(minutes_offsets[-1] + step)

    sub["timestamp"] = base_start + pd.to_timedelta(minutes_offsets, unit="m")

    # Write back into main df with original indices
    df_preprocessed.loc[sub.index, "timestamp"] = sub["timestamp"]

print("[INFO] Synthetic Timestamps Generated per Product with per-engine-type Frequency.")

# Sort globally and inspect
df_preprocessed = df_preprocessed.sort_values(["product_id", "timestamp"]).reset_index(drop=True)

print("[RESULT] Timestamp Range:",
      df_preprocessed["timestamp"].min(), "->", df_preprocessed["timestamp"].max())
print("[RESULT] Sample Rows:")
df_preprocessed.head()

[INFO] Input Rows: 10000 | Unique product_id: 10000


C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

[INFO] Synthetic Timestamps Generated per Product with per-engine-type Frequency.
[RESULT] Timestamp Range: 2024-01-01 06:00:00 -> 2024-04-30 18:00:00
[RESULT] Sample Rows:


C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  + pd.Timedelta(start_hour, "H")
C:\Users\Rahfi\AppData\Local\Temp\ipykernel_21888\1673206874.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

,unit_id,product_id,engine_type,air_temp,process_temp,rpm,torque_nm,tool_wear,target,failure_type,timestamp
0,11,H29424,H,298.4,308.9,1782,23.9,24,0,No Failure,2024-01-11 16:00:00
1,12,H29425,H,298.6,309.1,1423,44.3,29,0,No Failure,2024-03-20 11:00:00
2,19,H29432,H,298.8,309.2,1306,54.5,50,0,No Failure,2024-02-22 17:00:00
3,21,H29434,H,298.9,309.3,1375,42.7,58,0,No Failure,2024-01-11 15:00:00
4,28,H29441,H,299.1,309.4,1811,24.6,77,0,No Failure,2024-01-25 07:00:00


In [5]:
os.makedirs("preprocessed_data", exist_ok=True)
df_preprocessed.to_csv("preprocessed_data/timestamped_predictive_maintenance.csv", index=False)
print("[INFO] Timestamp Dataset Created and Saved Successfully")

[INFO] Timestamp Dataset Created and Saved Successfully
